In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/PhiUSIIL_Phishing_URL_Dataset.csv"
)

print(df.shape)

(235795, 56)


In [4]:
# 1 Load dataset

df = pd.read_csv(
    "../data/PhiUSIIL_Phishing_URL_Dataset.csv"
)

# Separate numeric features only
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
numeric = df.select_dtypes(include=[np.number]).columns.tolist()
numeric = [col for col in numeric if col != "label"]

X = df[numeric]
y = df["label"]

In [5]:
# 2 Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [6]:
print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

print("Jumlah fitur:", X_train.shape[1])

print("Distribusi kelas training:")
print(y_train.value_counts())

print("\nDistribusi kelas testing:")
print(y_test.value_counts())

Jumlah data training: 165056
Jumlah data testing: 70739
Jumlah fitur: 50
Distribusi kelas training:
label
1    94395
0    70661
Name: count, dtype: int64

Distribusi kelas testing:
label
1    40455
0    30284
Name: count, dtype: int64


In [7]:
# 3 Evaluation Pipeline

def evaluate(model, X_train, X_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall:", recall_score(y_test, preds))
    print("F1 Score:", f1_score(y_test, preds))
    print("ROC-AUC:", roc_auc_score(y_test, probs))


evaluate(LogisticRegression(max_iter=10000), X_train, X_test, "Logistic Regression")

evaluate(DecisionTreeClassifier(), X_train, X_test, "Decision Tree")

evaluate(RandomForestClassifier(n_estimators=200), X_train, X_test, "Random Forest")

evaluate(ExtraTreesClassifier(n_estimators=200), X_train, X_test, "Extra Trees")


=== Logistic Regression ===
Accuracy: 0.9999010446853928
Precision: 0.9998764150477037
Recall: 0.999950562353232
F1 Score: 0.9999134873258932
ROC-AUC: 0.9999976051701291

=== Decision Tree ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0

=== Random Forest ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0

=== Extra Trees ===
Accuracy: 0.9999717270529693
Precision: 0.9999505647971921
Recall: 1.0
F1 Score: 0.9999752817876211
ROC-AUC: 1.0


In [8]:
from pathlib import Path

from sklearn.base import clone

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

reference_models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=200),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200)
}

reproduction_results = []

for name, model in reference_models.items():

    model = clone(model)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    reproduction_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    })

reference_df = pd.DataFrame(reproduction_results)

reference_df.to_csv(
    RESULTS_DIR / "reference_reproduction_metrics.csv",
    index=False
)

print("Hasil reproduksi berhasil disimpan!")
display(reference_df.round(6))

Hasil reproduksi berhasil disimpan!


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.999901,0.999876,0.999951,0.999913,0.999998
1,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000
2,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000
3,Extra Trees,0.999972,0.999951,1.000000,0.999975,1.000000
